# Image Compression with the Singular Value Decomposition

**Python course final project**

---

## Contents
1. Introduction and problem statement
2. Mathematical background: the SVD
3. Low-rank approximation and the Eckart–Young–Mirsky theorem
4. How much do we save? Storage analysis
5. Algorithms: how to compute a truncated SVD
6. Experiments
7. Conclusions and limitations
8. References

## 1. Introduction and problem statement

A digital grayscale image is a grid of pixel intensities, i.e. a **matrix** $A \in \mathbb{R}^{m\times n}$,
with entries in $[0,1]$ (0 = black, 1 = white). A colour image is three such matrices (R, G, B).
A $512\times512$ photo already has $262\,144$ numbers.

But images are *not* random tables of numbers. Neighbouring pixels are strongly correlated,
large regions are smooth, and edges repeat along rows and columns. This means the matrix
contains a lot of **redundancy**, and redundancy can be removed.

> **Problem.** Given an image $A\in\mathbb{R}^{m\times n}$ and a storage budget, find a matrix
> $\hat A$ that can be stored with far fewer than $mn$ numbers and is *as close as possible* to $A$.
>
> 1. What is the **best possible** approximation when we restrict $\hat A$ to rank $k$?
> 2. How do we **compute** it efficiently, and how do our own implementations compare with a library?
> 3. How does this approach compare with the **DCT**, the transform behind JPEG?

"Close" needs a precise meaning. We use two matrix norms:

* the **Frobenius norm** $\|A\|_F=\big(\sum_{i,j}a_{ij}^2\big)^{1/2}$, the total pixel error
  (directly related to the MSE and PSNR used for images);
* the **spectral norm** $\|A\|_2=\max_{\|x\|=1}\|Ax\|$, the worst-case amplification of a vector.

The key tool is the **Singular Value Decomposition (SVD)**.

## 2. Mathematical background: the SVD

**Theorem (SVD).** Every real matrix $A\in\mathbb{R}^{m\times n}$ of rank $r$ can be written as
$$
A = U\Sigma V^\top=\sum_{i=1}^{r}\sigma_i\,u_i v_i^\top ,
$$
where $U=[u_1,\dots,u_m]\in\mathbb{R}^{m\times m}$ and $V=[v_1,\dots,v_n]\in\mathbb{R}^{n\times n}$ are
orthogonal, and $\Sigma\in\mathbb{R}^{m\times n}$ is diagonal with
$\sigma_1\ge\sigma_2\ge\dots\ge\sigma_r>0=\sigma_{r+1}=\dots$ The $\sigma_i$ are the **singular values**,
and $u_i, v_i$ are the left and right **singular vectors**.

**Proof (existence).** The matrix $A^\top A\in\mathbb{R}^{n\times n}$ is symmetric and positive
semi-definite, because $x^\top A^\top A x=\|Ax\|^2\ge 0$. By the spectral theorem it has an
orthonormal eigenbasis $v_1,\dots,v_n$ with eigenvalues $\lambda_1\ge\dots\ge\lambda_n\ge 0$.
Put $\sigma_i=\sqrt{\lambda_i}$. Note that $\|Av_i\|^2=v_i^\top A^\top A v_i=\lambda_i=\sigma_i^2$.

For every $i$ with $\sigma_i>0$ define $u_i = Av_i/\sigma_i$. These vectors are orthonormal:
$$
u_i^\top u_j=\frac{v_i^\top A^\top A v_j}{\sigma_i\sigma_j}=\frac{\lambda_j\, v_i^\top v_j}{\sigma_i\sigma_j}=\delta_{ij}.
$$
There are exactly $r=\operatorname{rank}A$ of them, since $\operatorname{rank}A^\top A=\operatorname{rank}A$.
Extend them to an orthonormal basis $u_1,\dots,u_m$ of $\mathbb{R}^m$. For $\sigma_i>0$ we have
$Av_i=\sigma_i u_i$ by construction. For $\sigma_i=0$ we have $\|Av_i\|=0$, so again $Av_i=0=\sigma_iu_i$.
In matrix form this reads $AV=U\Sigma$, and since $V$ is orthogonal, $A=U\Sigma V^\top$. $\blacksquare$

**Interpretation.** Every linear map is a *rotation* ($V^\top$), followed by an axis-wise *stretch* ($\Sigma$),
followed by another *rotation* ($U$). For an image, $A=\sum_i\sigma_i u_iv_i^\top$ writes the picture
as a sum of **rank-one "layers"** $u_iv_i^\top$ (an outer product of a column profile and a row
profile), each weighted by its importance $\sigma_i$.

Two facts we will need:

* **Orthogonal invariance:** $\|QAP\|_F=\|A\|_F$ and $\|QAP\|_2=\|A\|_2$ for orthogonal $Q,P$. Hence
  $\|A\|_F^2=\|\Sigma\|_F^2=\sum_i\sigma_i^2$ and $\|A\|_2=\sigma_1$.
* $\sigma_i^2$ are the eigenvalues of $A^\top A$. This is how we will compute them in Section 5.

## 3. Low-rank approximation and the Eckart–Young–Mirsky theorem

Keep only the first $k$ terms:
$$
A_k=\sum_{i=1}^{k}\sigma_i u_i v_i^\top = U_k\Sigma_kV_k^\top .
$$
$A_k$ has rank $k$. Since $A-A_k=\sum_{i>k}\sigma_iu_iv_i^\top$ is itself an SVD, orthogonal
invariance immediately gives the errors
$$
\|A-A_k\|_2=\sigma_{k+1},\qquad \|A-A_k\|_F=\Big(\sum_{i=k+1}^{r}\sigma_i^2\Big)^{1/2}.
$$

The remarkable fact is that **no other rank-$k$ matrix does better.**

**Theorem (Eckart–Young 1936, Mirsky 1960).** For every $B\in\mathbb{R}^{m\times n}$ with $\operatorname{rank}B\le k$:
$$
\|A-B\|_2\ \ge\ \sigma_{k+1}=\|A-A_k\|_2
\qquad\text{and}\qquad
\|A-B\|_F\ \ge\ \Big(\sum_{i>k}\sigma_i^2\Big)^{1/2}=\|A-A_k\|_F .
$$

**Proof: spectral norm.** Since $\operatorname{rank}B\le k$, the null space $\mathcal N(B)$ has dimension
$\ge n-k$. The subspace $\mathcal V=\operatorname{span}\{v_1,\dots,v_{k+1}\}$ has dimension $k+1$. Because
$(n-k)+(k+1)>n$, the two subspaces intersect non-trivially. Pick a unit vector $w=\sum_{i=1}^{k+1}c_iv_i$
in the intersection, so $\sum c_i^2=1$. Then $Bw=0$ and
$$
\|A-B\|_2^2\ \ge\ \|(A-B)w\|^2=\|Aw\|^2=\Big\|\sum_{i=1}^{k+1}c_i\sigma_iu_i\Big\|^2=\sum_{i=1}^{k+1}c_i^2\sigma_i^2\ \ge\ \sigma_{k+1}^2 .
$$

**Lemma (Weyl's inequality for singular values).** For any $X,Y$ and indices $i,j\ge1$:
$\ \sigma_{i+j-1}(X+Y)\le\sigma_i(X)+\sigma_j(Y)$.

*Proof.* Let $X_{i-1},Y_{j-1}$ be the truncated SVDs of $X,Y$ of ranks $i-1,j-1$. Their sum has rank
$\le i+j-2$. By the spectral part just proven (applied to $X+Y$ with $k=i+j-2$),
$$
\sigma_{i+j-1}(X+Y)\le\|X+Y-X_{i-1}-Y_{j-1}\|_2\le\|X-X_{i-1}\|_2+\|Y-Y_{j-1}\|_2=\sigma_i(X)+\sigma_j(Y).\ \square
$$

**Proof: Frobenius norm.** Apply Weyl with $X=A-B$, $Y=B$ and $j=k+1$. Since $\operatorname{rank}B\le k$,
$\sigma_{k+1}(B)=0$, so $\sigma_{i+k}(A)\le\sigma_i(A-B)$ for all $i\ge1$. Squaring and summing,
$$
\|A-B\|_F^2=\sum_{i\ge1}\sigma_i(A-B)^2\ \ge\ \sum_{i\ge1}\sigma_{i+k}(A)^2=\sum_{i>k}\sigma_i^2=\|A-A_k\|_F^2.\ \blacksquare
$$

**Consequences for images.**
* The **retained energy** $\ \mathcal E_k=\sum_{i\le k}\sigma_i^2\big/\sum_i\sigma_i^2\ $ gives the relative error exactly:
  $\|A-A_k\|_F/\|A\|_F=\sqrt{1-\mathcal E_k}$.
* The compressibility of an image is **entirely determined by how fast its singular values decay.**
  We test this in Experiment 1.
* The truncated SVD is optimal *among rank-$k$ matrices*. It says nothing about other kinds of
  representations, such as a sparse set of DCT coefficients. That is why the comparison with
  JPEG-style compression in Experiment 4 is interesting.

## 4. How much do we save? Storage analysis

To store $A_k$ we keep $U_k$ ($mk$ numbers), $V_k$ ($nk$) and $\sigma_1..\sigma_k$ ($k$), in total
$$
\text{cost}(k)=k(m+n+1)\quad\text{instead of}\quad mn,
\qquad\text{compression ratio}\ \ \rho(k)=\frac{mn}{k(m+n+1)} .
$$
Compression only pays off while $k(m+n+1)<mn$, i.e. below the **break-even rank**
$k^*=\big\lfloor (mn-1)/(m+n+1)\big\rfloor$. For a square $N\times N$ image $k^*\approx N/2$:
a $512\times512$ image must be approximated with $k<255$ to save anything at all.

A colour image is compressed channel by channel, which costs $3k(m+n+1)$ numbers.

(Here we count *numbers*. Real formats would also quantise them to fewer bits; we ignore that
for both SVD and DCT, so the comparison stays fair.)

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# make the repository root importable (works when started from notebook/ or the root)
ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT))

from svdcomp.io import load_image, rgb_to_gray
from svdcomp.compress import (numpy_svd, low_rank_approx, compress_image, storage_cost,
                              compression_ratio, break_even_rank, energy_retained,
                              theoretical_errors, rank_for_energy)
from svdcomp.power_svd import power_svd, power_svd_with_info, dominant_triplet
from svdcomp.randomized_svd import randomized_svd
from svdcomp.dct import dct_compress_image, block_dct
from svdcomp.metrics import psnr, ssim, relative_frobenius_error

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "image.cmap": "gray", "figure.figsize": (9, 4)})
IMG = ROOT / "images"
rng = np.random.default_rng(0)

In [ ]:
m = n = 512
print(f"break-even rank for {m}x{n}: k* = {break_even_rank(m, n)}")
for k in (5, 20, 50, 100, 200, 255, 256):
    print(f"k = {k:3d}: stored = {storage_cost(m, n, k):7d} values, ratio = {compression_ratio(m, n, k):6.2f}")